# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
- Answer questions using internal knowledge (RAG)
- Search the web when needed
- Maintain conversation state
- Return structured outputs
- Store useful information for future use

### 1. Setup

In [1]:
from lib.agents import Agent, AgentState
from lib.messages import UserMessage, SystemMessage, AIMessage#, ToolMessage
from lib.tooling import tool
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult
from lib.state_machine import Run
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager
from lib.tooling import Tool

import os
import time
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient  # pip install tavily-python


In [2]:
# Load environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### 2 Tools

#### 2.1 Retrieve Game Tool

In [3]:
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
@tool
def retrieve_game(query: str) -> dict:
    """A tool that performs a semantic search in the collection using the query.
    
    
    Args:
        query: A question about the game industry.
        
    Returns:
        A list of results, where each element contains:
        - Platform: like Game Boy, Playstation 5, Xbox 360...
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    """
    embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

    results = collection.query(embedding_fn.embed_query(query), n_results=5).get("metadatas")[0]  # Adjust n_results as needed
    return results
# retrieve_game("a game with cars")

#### 2.2 Evaluate Retrieval Tool

In [4]:
@tool
def evaluate_retrieval(question: str, retrieved_docs=None) -> dict:
    """A tool that evaluates the usability of the retrieved documents to respond to the user's question.
    
    Args:
        question: The original question from the user.
        retrieved_docs: A list of retrieved documents most similar to the user query in the Vector Database.
        
    Returns:
        A dictionary containing:
        - useful: A boolean indicating whether the documents are useful to answer the question.
        - description: A detailed explanation about the evaluation result.
    """
    # Here you would implement the logic to evaluate the documents using an LLM or other methods
    # For now, we will return a placeholder response

    llm = Agent(
        model_name="gpt-3.5-turbo", #"gpt-4o-mini",
        instructions="Your task is to evaluate if the documents are enough to respond to the query. Give a detailed explanation, so it's possible to take an action to accept it or not.",
        temperature=0.7,
    )
  
    input=[
        SystemMessage(
            content=(
                f"Your task is to evaluate if the documents are enough to respond to the query. Give a detailed explanation, so it's possible to take an action to accept it or not."
            )
        ),
        UserMessage(
            content=f"Question to investigate: {question}\nRetrieved Documents: {retrieved_docs}"
        ),
    ]
    
    query=str(input)
    response = llm.invoke(query=query)    

    test_case = TestCase(
            id="game_query_1",
            description="Evaluate the usability of retrieved documents for a game-related query.",
            user_query=str(input)  ,
            expected_tools=["retrieve_game"],
            reference_answer=None, 
            max_steps=4
    )
    
    evaluator = AgentEvaluator()
    black_box_eval:EvaluationResult = evaluator.evaluate_final_response(
        test_case=test_case,
        agent_response=response.get_final_state()["messages"][-1].content,
        execution_time=1.2,
        total_tokens=150
    )
    
    toanalyse=black_box_eval.model_dump()
    
    return {
        "useful": toanalyse.get('task_completion')['task_completed'], 
        "description": toanalyse.get('feedback')
    }
# evaluate_retrieval("a game with cars", retrieve_game("a game with cars"))

#### 2.3 Game Web Search Tool

In [5]:
@tool
def game_web_search(question):
    """A tool that performs a semantic search on the web using Tavily client.
    
    Args:
        question: A question about the game industry.
        
    Returns:
        A list of search results relevant to the question together with the URLs of the sources.
    """
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    results = tavily_client.search(question)
    return results
# game_web_search("tell me about gran tourismo")

#### 2.4 Long Term Memory Tools

In [6]:
db = VectorStoreManager(OPENAI_API_KEY)
ltm = LongTermMemory(db)

memory_register_tool = Tool(
    name="memory_register_tool",
    description="Store a user preference or important fact in long-term memory.",
    func=lambda content: LongTermMemory.register(
        ltm,
        MemoryFragment(
            content=content,
            owner="UdaPlay",
            namespace="conversation"
        )
    )
)

memory_search_tool = Tool(
    name="memory_search_tool",
    description="Search long-term memory for previously stored user preferences or facts.",
    func=lambda query_text: LongTermMemory.search(
        ltm,
        query_text=query_text,
        owner="UdaPlay",
        limit=3,
        timestamp_filter=None,
        namespace="conversation"
    )
)

### 3. Definitions

#### 3.1 Agent Definition

In [7]:
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        memory_register_tool,
        memory_search_tool,
        retrieve_game, 
        evaluate_retrieval, 
        game_web_search
    ],
    instructions="""
        [Role]
        You are 'UdaPlay', a video game research assistant who only answers questions about video games in a brief manner.
        [Goal]
        Provide a sufficient answer to questions and remember useful facts about games, developers, publishers, release dates, platforms, genres, and related topics.
        [Instructions]
        Start with  'memory_register_tool'.
        Then, retrieve relevant information from memory using 'memory_search_tool'.
        If the answer of 'evaluate_retrieval' based on 'memory_search_tool' is uncertain or missing, use 'retrieve_game'.
        If the answer of 'evaluate_retrieval' based on 'retrieve_game' is uncertain or missing, use 'game_web_search'.
        Cite all sources with helpful information among 'memory' (from 'memory_search_tool') ,   'database' (from 'retrieve_game'), and 'website(s)' (from 'game_web_search') and give a short, clear final answer based only on verified facts.

        [Tools]
        - 'memory_register_tool': save a preference or important fact to memory
        - 'memory_search_tool': find past facts or preferences from memory
        - 'retrieve_game': search the internal database
        - 'evaluate_retrieval': judge whether retrieved documents from 'retrieve_game' or internal memory from 'memory_search_tool' are useful
        - 'game_web_search': search the web when needed

        [Output format]
        Use this structure:
          answer: "Brief, factual answer based only on verified sources."
          sources: "Any combination of memory, database, and website URLs."
        All sources listed provide complementary pieces of information to the final answer.
        Sources whose used information is completely in another source should not be listed again in the "sources" array.
        
        Example:
          answer: Pigalle was released for PC in 2008, and you played it.
          sources: [memory, "www.test.de"]
        
           
    """,
    temperature=0.7
)

#### 3.2 Query Definition + extra Knowledge

In [8]:
# extra_knowledge="'Mortal Kombat X' was often played by me and I enjoyed it."
# query="Was 'Mortal Kombat X' released for PlayStation 5 and did I like it?"

# extra_knowledge="'Grand Theft Auto 4' was often played by me and I enjoyed it."
# query="Was 'Grand Theft Auto 4' released for PC and did I like it?"

extra_knowledge="'Marvel's Spider-Man' was often played by me and I enjoyed it."
query="What year was 'Marvel's Spider-Man' released for PlayStation 5 and did I like it?"



### 4. Run the Agent

In [9]:
# Invoke agent and measure execution time

start_time = time.time() # only needed for evaluation
agent.memory.reset()
run_object:Run = agent.invoke(extra_knowledge)  # add a memory fragment to the agent's long term memory
run_object:Run = agent.invoke(query)
execution_time = time.time() - start_time     # only needed for evaluation


# Get final state and response
final_state:AgentState = run_object.get_final_state()

print("extra_knowledge: ", extra_knowledge)
print("query: ", query) 
print(f"\nReply of agent after {execution_time:.2f} seconds:")
print(final_state["messages"][-1].content)
print("")




extra_knowledge:  'Marvel's Spider-Man' was often played by me and I enjoyed it.
query:  What year was 'Marvel's Spider-Man' released for PlayStation 5 and did I like it?

Reply of agent after 40.19 seconds:
answer: "'Marvel's Spider-Man Remastered' was released for PlayStation 5 on November 12, 2020. You enjoyed playing 'Marvel's Spider-Man'."
sources: ["www.wikipedia.org/wiki/Marvel%27s_Spider-Man_(video_game)", memory]



### (5. Evaluate the Agent)

In [10]:
#Evaluation of the agent's performance using the AgentEvaluator class

# Evaluation of the agent's performance using the AgentEvaluator class
# test_case = TestCase(
#     id="game_query_1",
#     description="Evaluate the usability of retrieved documents for a game-related query.",
#     user_query=query,
    
#     expected_tools=[
#             "memory_register_tool",
#             "memory_search_tool",
#             "retrieve_game", 
#             "evaluate_retrieval", 
#             "game_web_search"
#     ],
#     reference_answer=None, 
#     max_steps=10
# )

# evaluator = AgentEvaluator()
# if final_state and final_state.get("messages"):
#     # Find the last AI message as the final response
#     final_response = ""
#     for msg in reversed(final_state["messages"]):
#         if isinstance(msg, AIMessage) and msg.content:
#             final_response = msg.content
#             break
    
#     total_tokens = final_state.get("total_tokens", 0)
    
#     # Evaluate using all three methods (black box, single step, trajectory)
#     print("\n--- Black Box (Final Response) Evaluation ---")
#     black_box_eval:EvaluationResult = evaluator.evaluate_final_response(
#         test_case, final_response, execution_time, total_tokens
#     )
#     print(f"Overall Score: {black_box_eval.overall_score:.2f}")
#     print(f"Task Completed: {black_box_eval.task_completion.task_completed}")
#     print(f"Feedback: {black_box_eval.feedback}")
    
#     print("\n--- Single Step Evaluation ---")
#     step_eval:EvaluationResult = evaluator.evaluate_single_step(
#         final_state["messages"], test_case.expected_tools
#     )
#     print(f"Overall Score: {step_eval.overall_score:.2f}")
#     print(f"Correct Tool Selected: {step_eval.tool_interaction.correct_tool_selected}")
#     print(f"Feedback: {step_eval.feedback}")
    
#     print("\n--- Trajectory Evaluation ---")
#     traj_eval:EvaluationResult = evaluator.evaluate_trajectory(test_case, run_object)
#     print(f"Overall Score: {traj_eval.overall_score:.2f}")
#     print(f"Steps Taken: {traj_eval.task_completion.steps_taken}")
#     print(f"Total Tokens: {traj_eval.system_metrics.total_tokens}")
#     print(f"Execution Time: {traj_eval.system_metrics.execution_time:.2f}s")
#     print(f"Estimated Cost: ${traj_eval.system_metrics.cost_estimate:.6f}")
#     print(f"Feedback: {traj_eval.feedback}")
    
# else:
#     print("ERROR: No final state or messages found")